**Câu 1:**

In [73]:
import sqlite3
import math

conn = sqlite3.connect("bai_tap_chuong_03")
cursor = conn.cursor()

cursor.execute('DROP TABLE IF EXISTS du_lieu')
cursor.execute("""
    CREATE TABLE du_lieu (
        a FLOAT,
        b FLOAT
    )
""")

# Giả sử chúng ta có các cặp dữ liệu sau cho A và B
data = [
    (1.0, 2.0),
    (2.0, 3.0),
    (3.0, 5.0),
    (4.0, 4.0),
    (5.0, 6.0)
]

cursor.executemany("INSERT INTO du_lieu (a, b) VALUES (?, ?)", data)
conn.commit()

cursor.execute("SELECT COUNT(*) FROM du_lieu")
n = cursor.fetchone()[0]

# Tổng của a_i * b_i
cursor.execute("SELECT SUM(a * b) FROM du_lieu")
tong_ab = cursor.fetchone()[0]

# Tổng của a_i
cursor.execute("SELECT SUM(a) FROM du_lieu")
tong_a = cursor.fetchone()[0]

# Tổng của b_i
cursor.execute("SELECT SUM(b) FROM du_lieu")
tong_b = cursor.fetchone()[0]

# Tổng bình phương của a_i
cursor.execute("SELECT SUM(a * a) FROM du_lieu")
tong_a_binh_phuong = cursor.fetchone()[0]

# Tổng bình phương của b_i
cursor.execute("SELECT SUM(b * b) FROM du_lieu")
tong_b_binh_phuong = cursor.fetchone()[0]

# Tính hệ số tương quan r_AB theo công thức
tu_so = n * tong_ab - tong_a * tong_b
mau_so = math.sqrt(n * tong_a_binh_phuong - tong_a**2) * math.sqrt(n * tong_b_binh_phuong - tong_b**2)

if mau_so == 0:
    print("Không thể tính hệ số tương quan: mẫu số bằng 0.")
else:
    r_AB = tu_so / mau_so
    print(f"Hệ số tương quan r_AB là: {r_AB:.4f}")

Hệ số tương quan r_AB là: 0.9000


=> Hệ số tương quan r_AB nằm trong khoảng từ -1 đến 1. Giá trị gần 1 (kết quả là 0.9) cho thấy A & B có mối tương quan dương mạnh.

**Câu 2:**

In [74]:
cursor.execute('DROP TABLE IF EXISTS diem_so')
cursor.execute("""
    CREATE TABLE diem_so (
        ngay TEXT,
        A FLOAT,
        B FLOAT,
        C FLOAT
    )
""")

du_lieu = [
    ("Day 1", 8.0, 9.0, 7.0),
    ("Day 2", 7.5, 8.5, 7.0),
    ("Day 3", 6.0, 7.0, 8.0),
    ("Day 4", 7.0, 6.0, 5.0)
]

cursor.executemany("INSERT INTO diem_so (ngay, A, B, C) VALUES (?, ?, ?, ?)", du_lieu)
conn.commit()

# Tính variance cho từng ngày và trung bình
variances = []

for day in ["Day 1", "Day 2", "Day 3", "Day 4"]:
    cursor.execute("SELECT A, B, C FROM diem_so WHERE ngay = ?", (day,))
    scores = cursor.fetchone()
    
    # Tính trung bình của điểm số trong ngày
    mean = sum(scores) / len(scores)
    
    # Tính variance
    variance = sum((x - mean) ** 2 for x in scores) / len(scores)
    variances.append((day, variance))
    print(f"Độ chênh lệch (variance) của {day}: {variance:.4f}")

# Tính trung bình của các variance
average_variance = sum(v for _, v in variances) / len(variances)
print(f"Độ chênh lệch trung bình của 4 ngày: {average_variance:.4f}")

# Xác định ngày phù hợp nhất (variance nhỏ nhất)
min_variance_day = min(variances, key=lambda x: x[1])
print(f"Ngày phù hợp nhất là {min_variance_day[0]} với độ chênh lệch: {min_variance_day[1]:.4f}")

# Chuyển đổi dữ liệu sang dạng quan hệ
cursor.execute('DROP TABLE IF EXISTS du_lieu_quan_he')
cursor.execute("""
    CREATE TABLE du_lieu_quan_he (
        ngay TEXT,
        mau TEXT,
        diem_so FLOAT
    )
""")

du_lieu_quan_he = []
for day, a, b, c in du_lieu:
    du_lieu_quan_he.append((day, "A", a))
    du_lieu_quan_he.append((day, "B", b))
    du_lieu_quan_he.append((day, "C", c))

cursor.executemany("INSERT INTO du_lieu_quan_he (ngay, mau, diem_so) VALUES (?, ?, ?)", du_lieu_quan_he)
conn.commit()

# Phân loại điểm số thành các nhóm (Thấp, Trung bình, Cao)
cursor.execute('DROP TABLE IF EXISTS du_lieu_phan_loai')
cursor.execute("""
    CREATE TABLE du_lieu_phan_loai AS
    SELECT ngay, mau, diem_so,
           CASE
               WHEN diem_so < 7 THEN 'Thap'
               WHEN diem_so >= 7 AND diem_so < 8 THEN 'Trung_binh'
               ELSE 'Cao'
           END AS nhom
    FROM du_lieu_quan_he
""")

cursor.execute("""
    SELECT ngay, nhom, COUNT(*) as so_luong
    FROM du_lieu_phan_loai
    GROUP BY ngay, nhom
""")

contingency_table = {}
days = ["Day 1", "Day 2", "Day 3", "Day 4"]
groups = ["Thap", "Trung_binh", "Cao"]

for day, group, count in cursor.fetchall():
    if day not in contingency_table:
        contingency_table[day] = {}
    contingency_table[day][group] = count

for day in days:
    if day not in contingency_table:
        contingency_table[day] = {}
    for group in groups:
        if group not in contingency_table[day]:
            contingency_table[day][group] = 0

print("\nBảng tần số:")
print("Ngày\t\tThấp\tTB\tCao")
for day in days:
    print(f"{day}\t\t{contingency_table[day]['Thap']}\t{contingency_table[day]['Trung_binh']}\t{contingency_table[day]['Cao']}")

row_totals = {day: sum(contingency_table[day].values()) for day in days}
col_totals = {"Thap": 0, "Trung_binh": 0, "Cao": 0}
for day in days:
    for group in groups:
        col_totals[group] += contingency_table[day][group]

grand_total = sum(row_totals.values())

expected = {}
for day in days:
    expected[day] = {}
    for group in groups:
        expected[day][group] = (row_totals[day] * col_totals[group]) / grand_total

chi_squared = 0
for day in days:
    for group in groups:
        observed = contingency_table[day][group]
        exp = expected[day][group]
        if exp > 0:  # Tránh chia cho 0
            chi_squared += (observed - exp) ** 2 / exp

df = (len(days) - 1) * (len(groups) - 1)

print(f"\nGiá trị Chi-squared: {chi_squared:.4f}")
print(f"Bậc tự do (degrees of freedom): {df}")

critical_value = 12.592  
if chi_squared > critical_value:
    print("Kết luận: Có sự phụ thuộc giữa ngày và nhóm điểm số (bác bỏ giả thuyết không).")
else:
    print("Kết luận: Không có đủ bằng chứng để nói rằng ngày và nhóm điểm số phụ thuộc vào nhau.")

Độ chênh lệch (variance) của Day 1: 0.6667
Độ chênh lệch (variance) của Day 2: 0.3889
Độ chênh lệch (variance) của Day 3: 0.6667
Độ chênh lệch (variance) của Day 4: 0.6667
Độ chênh lệch trung bình của 4 ngày: 0.5972
Ngày phù hợp nhất là Day 2 với độ chênh lệch: 0.3889

Bảng tần số:
Ngày		Thấp	TB	Cao
Day 1		0	1	2
Day 2		0	2	1
Day 3		1	1	1
Day 4		2	1	0

Giá trị Chi-squared: 6.2667
Bậc tự do (degrees of freedom): 6
Kết luận: Không có đủ bằng chứng để nói rằng ngày và nhóm điểm số phụ thuộc vào nhau.


X^2 = 6.9333 < 12.592, ta không bác bỏ giả thuyết H0, không có đủ bằng chứng để kết luận rằng ngày và nhóm điểm số phụ thuộc vào nhau.

**Câu 3:**

In [75]:
cursor.execute('DROP TABLE IF EXISTS flights')
cursor.execute("""
    CREATE TABLE flights (
        departure_time INTEGER
    )
""")

du_lieu = [
    (830,),
    (1445,),
    (930,),
    (1545,)
]

cursor.executemany("INSERT INTO flights (departure_time) VALUES (?)", du_lieu)
conn.commit()

def convert_time(time_int):
    time_str = str(time_int).zfill(4)  
    
    hours = int(time_str[:2])
    minutes = int(time_str[2:])
    
    period = "AM" if hours < 12 else "PM"
    
    if hours == 0:
        hours = 12  
    elif hours > 12:
        hours -= 12  
    
    return f"{hours:02d}:{minutes:02d} {period}"

cursor.execute("ALTER TABLE flights ADD COLUMN formatted_time TEXT")

cursor.execute("SELECT departure_time FROM flights")
rows = cursor.fetchall()

for row in rows:
    time_int = row[0]
    formatted_time = convert_time(time_int)
    cursor.execute("UPDATE flights SET formatted_time = ? WHERE departure_time = ?", (formatted_time, time_int))

conn.commit()

cursor.execute("SELECT departure_time, formatted_time FROM flights")
results = cursor.fetchall()

print("Dạng số nguyên\tDạng thời gian")
for row in results:
    print(f"{row[0]}\t\t{row[1]}")

Dạng số nguyên	Dạng thời gian
830		08:30 AM
1445		02:45 PM
930		09:30 AM
1545		03:45 PM


**Câu 4:**

In [76]:
import statistics

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE du_lieu (
        gia_tri FLOAT
    )
""")

du_lieu_mau = [
    (1.0,),
    (2.0,),
    (3.0,),
    (4.0,),
    (5.0,),
    (10.0,),  
    (20.0,),  
    (2.5,),
    (3.5,),
    (4.5,)
]

cursor.executemany("INSERT INTO du_lieu (gia_tri) VALUES (?)", du_lieu_mau)
conn.commit()

# Tính trung vị của tập dữ liệu
cursor.execute("SELECT gia_tri FROM du_lieu")
values = [row[0] for row in cursor.fetchall()]
median = statistics.median(values)
print(f"Trung vị (median) của tập dữ liệu: {median}")

# Tính độ lệch tuyệt đối so với trung vị
cursor.execute("DROP TABLE IF EXISTS do_lech")
cursor.execute("""
    CREATE TABLE do_lech (
        gia_tri FLOAT,
        do_lech_tuyet_doi FLOAT
    )
""")

# Tính độ lệch tuyệt đối
for value in values:
    abs_deviation = abs(value - median)
    cursor.execute("INSERT INTO do_lech (gia_tri, do_lech_tuyet_doi) VALUES (?, ?)", (value, abs_deviation))

conn.commit()

# Tính MAD 
cursor.execute("SELECT do_lech_tuyet_doi FROM do_lech")
abs_deviations = [row[0] for row in cursor.fetchall()]
mad = statistics.median(abs_deviations)
print(f"MAD (Median Absolute Deviation): {mad}")

# Xác định các giá trị ngoại lệ theo yêu cầu (lớn hơn 1.5 lần MAD)
threshold = 1.5 * mad
print(f"Ngưỡng: 1.5 * MAD = {threshold}")

# Tìm các giá trị ngoại lệ
cursor.execute("""
    SELECT gia_tri
    FROM do_lech
    WHERE do_lech_tuyet_doi > ?
""", (threshold,))

outliers = cursor.fetchall()
if outliers:
    print(f"Các giá trị ngoại lệ (có độ lệch tuyệt đối > {threshold}):")
    for outlier in outliers:
        print(f"- {outlier[0]}")
else:
    print("Không có giá trị ngoại lệ.")

Trung vị (median) của tập dữ liệu: 3.75
MAD (Median Absolute Deviation): 1.25
Ngưỡng: 1.5 * MAD = 1.875
Các giá trị ngoại lệ (có độ lệch tuyệt đối > 1.875):
- 1.0
- 10.0
- 20.0


**Câu 5:**

In [77]:
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE Patient (
        last_name TEXT,
        weight FLOAT,
        height FLOAT
    )
""")

du_lieu_mau = [
    ("Smith", 70.5, 170.0),
    ("Smith", 70.5, 165.0),  
    ("Johnson", 80.0, 180.0),
    ("Smith", 72.0, 168.0),  
    ("Johnson", 80.0, 175.0),  
]

cursor.executemany("INSERT INTO Patient (last_name, weight, height) VALUES (?, ?, ?)", du_lieu_mau)
conn.commit()

cursor.execute("CREATE TABLE Patient_with_id AS SELECT rowid AS id, * FROM Patient")
conn.commit()

cursor.execute("SELECT id, last_name, weight FROM Patient_with_id")
records = cursor.fetchall()

total_pairs = 0
matching_pairs = 0  

for i in range(len(records)):
    for j in range(i + 1, len(records)):
        record1 = records[i]
        record2 = records[j]
        
        distance = 0
        if record1[1] != record2[1]:
            distance += 1
        if record1[2] != record2[2]:
            distance += 1
        
        total_pairs += 1
        if distance == 0:
            matching_pairs += 1
        
        result = "Đây có thể là một người" if distance == 0 else "Đây là hai người khác nhau"
        print(f"Cặp ({record1[0]}, {record2[0]}): last_name: {record1[1]} vs {record2[1]}, weight: {record1[2]} vs {record2[2]} -> Khoảng cách Boolean: {distance} -> {result}")

print(f"\nTổng số cặp bản ghi: {total_pairs}")
print(f"Số cặp có khoảng cách Boolean = 0 (giống nhau): {matching_pairs}")
matching_ratio = matching_pairs / total_pairs if total_pairs > 0 else 0
print(f"Tỷ lệ cặp giống nhau: {matching_ratio:.2%}")

Cặp (1, 2): last_name: Smith vs Smith, weight: 70.5 vs 70.5 -> Khoảng cách Boolean: 0 -> Đây có thể là một người
Cặp (1, 3): last_name: Smith vs Johnson, weight: 70.5 vs 80.0 -> Khoảng cách Boolean: 2 -> Đây là hai người khác nhau
Cặp (1, 4): last_name: Smith vs Smith, weight: 70.5 vs 72.0 -> Khoảng cách Boolean: 1 -> Đây là hai người khác nhau
Cặp (1, 5): last_name: Smith vs Johnson, weight: 70.5 vs 80.0 -> Khoảng cách Boolean: 2 -> Đây là hai người khác nhau
Cặp (2, 3): last_name: Smith vs Johnson, weight: 70.5 vs 80.0 -> Khoảng cách Boolean: 2 -> Đây là hai người khác nhau
Cặp (2, 4): last_name: Smith vs Smith, weight: 70.5 vs 72.0 -> Khoảng cách Boolean: 1 -> Đây là hai người khác nhau
Cặp (2, 5): last_name: Smith vs Johnson, weight: 70.5 vs 80.0 -> Khoảng cách Boolean: 2 -> Đây là hai người khác nhau
Cặp (3, 4): last_name: Johnson vs Smith, weight: 80.0 vs 72.0 -> Khoảng cách Boolean: 2 -> Đây là hai người khác nhau
Cặp (3, 5): last_name: Johnson vs Johnson, weight: 80.0 vs 80.0 -